In [ ]:
with client_month_grid as (

    select
        cc.client_id,
        cc.campaigns_cnt,
        cc.first_campaign_dt::date as first_campaign_dt,

        gs.month_shift as month_shift,

        (
            date_trunc('month', cc.first_campaign_dt)::date
            + (gs.month_shift || ' month')::interval
        )::date as month_dt

    from cvm_sbx.{prefix}_CVMB_24118_client_cohorts cc

    cross join generate_series(-1, 4) as gs(month_shift)

    where cc.first_campaign_dt is not null

),

client_month_spend as (

    select
        cmg.client_id,
        cmg.campaigns_cnt,
        cmg.first_campaign_dt,
        cmg.month_shift,
        cmg.month_dt,

        coalesce(sum(ch.summ_discounted), 0) as month_spend

    from client_month_grid cmg

    left join dm.cheque ch
        on ch.contact_id = cmg.client_id
        and ch.datetime >= cmg.month_dt
        and ch.datetime < cmg.month_dt + interval '1 month'
        and ch.operation_type_id = 1
        and ch.summ_discounted > 0

    group by
        cmg.client_id,
        cmg.campaigns_cnt,
        cmg.first_campaign_dt,
        cmg.month_shift,
        cmg.month_dt

),

spend_iqr as (

    select
        campaigns_cnt,
        month_shift,

        percentile_cont(0.25)
            within group (order by month_spend) as q1,

        percentile_cont(0.75)
            within group (order by month_spend) as q3

    from client_month_spend

    group by
        campaigns_cnt,
        month_shift

),

client_month_spend_clean as (

    select
        cms.*

    from client_month_spend cms

    join spend_iqr iqr
        on cms.campaigns_cnt = iqr.campaigns_cnt
        and cms.month_shift = iqr.month_shift

    where cms.month_spend between
        iqr.q1 - 1.5 * (iqr.q3 - iqr.q1)
        and
        iqr.q3 + 1.5 * (iqr.q3 - iqr.q1)

)

select
    campaigns_cnt,
    month_shift,
    month_dt,

    case
        when month_shift = -1 then 'month_before_first_campaign'
        when month_shift = 0 then 'first_campaign_month'
        else 'month_plus_' || month_shift
    end as month_label,

    round(avg(month_spend), 2) as avg_spend_per_client,

    round(
        percentile_cont(0.5)
            within group (order by month_spend),
        2
    ) as median_spend_per_client,

    round(sum(month_spend), 2) as total_spend,

    count(*) as clients_after_cleaning

from client_month_spend_clean

group by
    campaigns_cnt,
    month_shift,
    month_dt

order by
    campaigns_cnt,
    month_shift,
    month_dt;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# df = pd.read_csv('rto_by_cohorts.csv')
# или df = pd.read_sql(query, con)

df['month_dt'] = pd.to_datetime(df['month_dt'])
df['campaigns_cnt'] = df['campaigns_cnt'].astype(int)
df = df.sort_values(['campaigns_cnt', 'month_dt'])


def format_y_axis():
    plt.gca().yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'.replace(',', ' '))
    )


# 1. Медианный РТО

plt.figure(figsize=(12, 6))

 for_cohorts = sorted(df['campaigns_cnt'].unique())

for cohort in for_cohorts:
    part = df[df['campaigns_cnt'] == cohort]

    plt.plot(
        part['month_dt'],
        part['median_spend_per_client'],
        marker='o',
        label=f'{cohort} камп.'
    )

plt.title('Медианный РТО на клиента по когортам')
plt.xlabel('Месяц')
plt.ylabel('Медианный РТО на клиента')
plt.grid(True, alpha=0.3)
plt.legend(title='Кол-во кампаний')
format_y_axis()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 2. Средний РТО

plt.figure(figsize=(12, 6))

for cohort in for_cohorts:
    part = df[df['campaigns_cnt'] == cohort]

    plt.plot(
        part['month_dt'],
        part['avg_spend_per_client'],
        marker='o',
        label=f'{cohort} камп.'
    )

plt.title('Средний РТО на клиента по когортам')
plt.xlabel('Месяц')
plt.ylabel('Средний РТО на клиента')
plt.grid(True, alpha=0.3)
plt.legend(title='Кол-во кампаний')
format_y_axis()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 3. Количество клиентов после очистки

plt.figure(figsize=(12, 6))

for cohort in for_cohorts:
    part = df[df['campaigns_cnt'] == cohort]

    plt.plot(
        part['month_dt'],
        part['clients_after_cleaning'],
        marker='o',
        label=f'{cohort} камп.'
    )

plt.title('Количество клиентов после очистки выбросов')
plt.xlabel('Месяц')
plt.ylabel('Количество клиентов')
plt.grid(True, alpha=0.3)
plt.legend(title='Кол-во кампаний')
format_y_axis()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()